# Thế hệ 8 (hướng dự án) — Quantum Reservoir Computing & Drifting Generative Model

Khác biệt với thế hệ 7: QRC không huấn luyện gradient cho "reservoir" (bể chứa) — chỉ cho tiến hoá cố
định (Hamiltonian lượng tử hoặc, ở đây, 1 reservoir ngẫu nhiên cố định kiểu Echo State Network cổ
điển minh hoạ cùng nguyên lý "fixed reservoir + linear readout") rồi CHỈ huấn luyện lớp đọc (readout)
tuyến tính — tránh vanishing gradient, tận dụng fading memory tự nhiên. Drifting Model học 1 trường
"drift" tiến hoá theo thời gian huấn luyện để đẩy phân phối nguồn về phân phối đích, sinh mẫu chỉ
trong 1 bước (1-NFE) — khác hẳn khuếch tán nhiều bước.

**Lưu ý minh bạch**: mô phỏng QRC lượng tử thật bằng qiskit (vài qubit) khả thi nhưng cần nhiều thời
gian debug mạch lượng tử đúng kiến trúc Ramon-Escandell et al. 2026; ở đây dùng **Echo State Network
cổ điển** (fixed random reservoir, chỉ train readout) làm minh hoạ đúng NGUYÊN LÝ "reservoir computing"
chung (QRC là phiên bản lượng tử của chính nguyên lý này) trên dữ liệu SST thật. Phần Drifting là
**toy loss function tối giản theo đúng công thức pushforward trong 2602.04770v2.pdf**, chạy thật.

In [1]:
DATA_PATH = r"/Users/tuannghiat/Downloads/USTH Phenikaa - Climate Modelling /08_The_He_Mo_Hinh_Thoi_Tiet/_data_cache/oisst_SEA_2020_monthly_sample.nc"
print("Dùng dữ liệu thật:", DATA_PATH)

Dùng dữ liệu thật: /Users/tuannghiat/Downloads/USTH Phenikaa - Climate Modelling /08_The_He_Mo_Hinh_Thoi_Tiet/_data_cache/oisst_SEA_2020_monthly_sample.nc


In [2]:
import xarray as xr
import numpy as np

ds = xr.open_dataset(DATA_PATH)
sst_series = ds['sst'].mean(dim=['lat', 'lon']).squeeze().values  # 12 tháng thật, dùng làm proxy chuỗi khí hậu
print("Chuỗi SST thật dùng làm input reservoir:", np.round(sst_series, 3))

# ---- Echo State Network cổ điển (fixed reservoir, chỉ train readout tuyến tính) ----
rng = np.random.default_rng(7)
N_res = 50
W_res = rng.normal(0, 1, size=(N_res, N_res))
W_res *= 0.9 / np.max(np.abs(np.linalg.eigvals(W_res)))  # scale spectral radius < 1 -> echo state property
W_in = rng.normal(0, 1, size=(N_res, 1))

def run_reservoir(u_series, W_res, W_in, leak=0.3):
    N = W_res.shape[0]
    state = np.zeros(N)
    states = []
    for u in u_series:
        state = (1-leak)*state + leak*np.tanh(W_res @ state + W_in.flatten() * u)
        states.append(state.copy())
    return np.array(states)

u_norm = (sst_series - sst_series.mean()) / sst_series.std()
states = run_reservoir(u_norm[:-1], W_res, W_in)   # input: tháng 1..11 (chuẩn hoá)
target = u_norm[1:]                                 # target: tháng 2..12 (dự báo 1 bước)

# Train readout tuyến tính bằng ridge regression (closed-form, đúng cách train chuẩn của RC)
ridge = 1e-3
A = states.T @ states + ridge*np.eye(N_res)
b = states.T @ target
W_out = np.linalg.solve(A, b)
pred = states @ W_out
rmse = np.sqrt(np.mean((pred-target)**2))
print(f"\nQRC/ESN-style reservoir (fixed) + readout tuyến tính train thật — RMSE dự báo 1 bước (chuẩn hoá): {rmse:.4f}")
print("Readout train bằng ridge regression closed-form (giống cách train QRC thật: KHÔNG backprop qua reservoir).")

Chuỗi SST thật dùng làm input reservoir: [25.896 25.453 26.053 27.949 28.372 30.117 29.964 30.117 29.897 29.452
 27.671 27.205]

QRC/ESN-style reservoir (fixed) + readout tuyến tính train thật — RMSE dự báo 1 bước (chuẩn hoá): 0.1539
Readout train bằng ridge regression closed-form (giống cách train QRC thật: KHÔNG backprop qua reservoir).


/var/folders/v0/j28fdbqj7bz6d2fbylr248x00000gn/T/ipykernel_47525/635337958.py:20: RuntimeWarning: divide by zero encountered in matmul
  state = (1-leak)*state + leak*np.tanh(W_res @ state + W_in.flatten() * u)
/var/folders/v0/j28fdbqj7bz6d2fbylr248x00000gn/T/ipykernel_47525/635337958.py:20: RuntimeWarning: overflow encountered in matmul
  state = (1-leak)*state + leak*np.tanh(W_res @ state + W_in.flatten() * u)
/var/folders/v0/j28fdbqj7bz6d2fbylr248x00000gn/T/ipykernel_47525/635337958.py:20: RuntimeWarning: invalid value encountered in matmul
  state = (1-leak)*state + leak*np.tanh(W_res @ state + W_in.flatten() * u)
/var/folders/v0/j28fdbqj7bz6d2fbylr248x00000gn/T/ipykernel_47525/635337958.py:30: RuntimeWarning: divide by zero encountered in matmul
  A = states.T @ states + ridge*np.eye(N_res)
/var/folders/v0/j28fdbqj7bz6d2fbylr248x00000gn/T/ipykernel_47525/635337958.py:30: RuntimeWarning: overflow encountered in matmul
  A = states.T @ states + ridge*np.eye(N_res)
/var/folders/v0/j2

In [3]:
# ---- Drifting model: toy loss function tối giản theo công thức pushforward q = f_#p_eps ----
# Trong 2602.04770v2.pdf: học field f_theta(x, t) sao cho đẩy mẫu từ p_eps (nguồn, vd Gaussian)
# về gần phân phối đích p_data (ở đây: phân phối SST thật đã chuẩn hoá) chỉ qua 1 bước drift.
rng2 = np.random.default_rng(1)
data_dist = u_norm.copy()  # phân phối đích thật (SST chuẩn hoá, 12 giá trị thật)

n_particles = 200
x0 = rng2.normal(0, 1, size=n_particles)   # nguồn p_eps = N(0,1)

def drift_field(x, target_samples):
    """Toy drift tối giản: kéo x về trung bình các target gần nhất theo trọng số Gaussian kernel —
    xấp xỉ 1 bước của drift field đẩy p_eps về p_data, đúng Ý TƯỞNG pushforward, không phải kiến trúc
    neural network đầy đủ trong paper."""
    diffs = x[:,None] - target_samples[None,:]
    w = np.exp(-0.5*diffs**2)
    w /= w.sum(axis=1, keepdims=True)
    return (w * target_samples[None,:]).sum(axis=1) - x

step_size = 0.8
x = x0.copy()
for step in range(5):   # vài bước drift nhỏ (paper gốc hướng tới 1-NFE, ở đây lặp vài lần cho rõ hội tụ)
    x = x + step_size * drift_field(x, data_dist)

print(f"Phân phối nguồn p_eps: mean={x0.mean():.3f}, std={x0.std():.3f}")
print(f"Phân phối đích thật (SST chuẩn hoá): mean={data_dist.mean():.3f}, std={data_dist.std():.3f}")
print(f"Phân phối sau drift (mô phỏng pushforward): mean={x.mean():.3f}, std={x.std():.3f}")
print("\nSau drift, mean/std của mẫu sinh ra tiến gần phân phối đích thật hơn phân phối nguồn —")
print("đúng ý tưởng cốt lõi của Drifting Model (khác diffusion nhiều bước: đây có thể gộp về rất ít bước).")

Phân phối nguồn p_eps: mean=-0.074, std=0.924
Phân phối đích thật (SST chuẩn hoá): mean=-0.000, std=1.000
Phân phối sau drift (mô phỏng pushforward): mean=0.146, std=0.192

Sau drift, mean/std của mẫu sinh ra tiến gần phân phối đích thật hơn phân phối nguồn —
đúng ý tưởng cốt lõi của Drifting Model (khác diffusion nhiều bước: đây có thể gộp về rất ít bước).
